Purpose of this notebook:
- testing out the Wikidata query service with the python sparqlwrapper library 
- using/adjusting the code from this [gitlab jupyter notebook](https://gitlab.wikimedia.org/-/snippets/257)

In [13]:
## import libraries

from SPARQLWrapper import SPARQLWrapper, JSON 
import polars as pl
import requests
import pandas as pd
import time
import datetime
from email.utils import parsedate_to_datetime

In [2]:
# Set configs

url = "https://query.wikidata.org/sparql"
user_agent = 'kg-digital-humanities/0.1.0 (https://github.com/FabiLochner/kg-digital-humanities; fabian.lochner@uni-oldenburg.de)'

### 1) Use requests library to debug 429 error codes

In [ ]:
# Query from the wikidata tutorial in notebooks/tutorial_wikidata_sparql.ipynb
query = """
# Brazilians who are poets 
SELECT ?item ?itemLabel WHERE {
  ?item wdt:P31 wd:Q5. # humans
  ?item wdt:P27 wd:Q155. # citizenship - Brazil
  ?item  wdt:P106 wd:Q49757. #occupation - poet
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
}
LIMIT 10
"""

In [9]:
resp = requests.get(
    url, 
    params= {"query": query, "format": "json"},
    headers = {"User-Agent": user_agent, "Accept": "application/sparql-results+json"}

)

print(f"Status: {resp.status_code}")
print(f"Retry-After: {resp.headers.get('Retry-After')}")
print(f"Body preview: {resp.text[:500]}")

Status: 200
Retry-After: None
Body preview: {
  "head" : {
    "vars" : [ "item", "itemLabel" ]
  },
  "results" : {
    "bindings" : [ {
      "item" : {
        "type" : "uri",
        "value" : "http://www.wikidata.org/entity/Q13012"
      },
      "itemLabel" : {
        "xml:lang" : "en",
        "type" : "literal",
        "value" : "Guimarães Rosa"
      }
    }, {
      "item" : {
        "type" : "uri",
        "value" : "http://www.wikidata.org/entity/Q23945"
      },
      "itemLabel" : {
        "xml:lang" : "en",
        "typ


In [ ]:
# Convert json output into pandas df

data = resp.json()

# Keys are defined by W3C SPARQL 1.1 Query Results JSON Format
# columns: item | itemLabel
rows = [
    {var: binding[var]["value"] for var in binding}
    for binding in data["results"]["bindings"]
]

df = pd.DataFrame(rows)
print(df.shape)
df.head()

(10, 2)


,item,itemLabel
0,http://www.wikidata.org/entity/Q13012,Guimarães Rosa
1,http://www.wikidata.org/entity/Q23945,Ana Cristina Cesar
2,http://www.wikidata.org/entity/Q184440,Jorge Amado
3,http://www.wikidata.org/entity/Q189042,Rubem Alves
4,http://www.wikidata.org/entity/Q199644,Adolfo Caminha


In [ ]:
# columns: item.type | item.value | itemLabel.type | itemLabel.value | itemLabel.xml:lang
df = pd.json_normalize(data["results"]["bindings"])
df.head()

,item.type,item.value,itemLabel.xml:lang,itemLabel.type,itemLabel.value
0,uri,http://www.wikidata.org/entity/Q13012,en,literal,Guimarães Rosa
1,uri,http://www.wikidata.org/entity/Q23945,en,literal,Ana Cristina Cesar
2,uri,http://www.wikidata.org/entity/Q184440,en,literal,Jorge Amado
3,uri,http://www.wikidata.org/entity/Q189042,en,literal,Rubem Alves
4,uri,http://www.wikidata.org/entity/Q199644,en,literal,Adolfo Caminha


### 2) Trying out the example of the newbook

In [12]:
sparql = SPARQLWrapper(url, agent = user_agent)

# Query from the wikidata tutorial in notebooks/tutorial_wikidata_sparql.ipynb
sparql.setQuery(query = """
# Brazilians who are poets 
SELECT ?item ?itemLabel WHERE {
  ?item wdt:P31 wd:Q5. # humans
  ?item wdt:P27 wd:Q155. # citizenship - Brazil
  ?item  wdt:P106 wd:Q49757. #occupation - poet
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
}
LIMIT 10
""")
sparql.setReturnFormat(JSON)
results = sparql.query().convert()

HTTPError: HTTP Error 429: Aggressively rate-limiting to 1 req / min - this rule was created during active wdqs outage (797a132)

- [Github Wikidata two phase extractor](https://gist.github.com/ArloDune/117ee69e72c1ceaae1c45818e5d646fa)
- [Official wikidata rate limits](https://www.mediawiki.org/wiki/Wikidata_Query_Service/User_Manual#Query_limits)

In [18]:
# Trying to debug the causes
## query is exactly the same
## user_agent is also the same:

sparql = SPARQLWrapper(url, agent=user_agent)
print(sparql.agent)  

kg-digital-humanities/0.1.0 (https://github.com/FabiLochner/kg-digital-humanities; fabian.lochner@uni-oldenburg.de)


### 3) Trying out own sparql project query with requests

#### 3.1) 1-Phase Architecture (Inefficient for bulk extraction)

In [ ]:
# Query from the wikidata tutorial in notebooks/tutorial_wikidata_sparql.ipynb
query_1 = """
# B20 German writers OR novelists OR poets with some exammple properties
SELECT ?item ?itemLabel ?sexLabel ?dobLabel ?dodLabel 
WHERE {
  ?item wdt:P31 wd:Q5. # humans
  
  # German citizenship OR writes in German language 
  {?item wdt:P27 wd:Q183.} # citizenship - Germany
  UNION
  {?item wdt:P6886 wd:Q188.} # writing_language - German
  
  # occupation: writer OR novelist OR poet                                  
  {?item  wdt:P106 wd:Q36180.} #occupation - writers
  UNION
  {?item  wdt:P106 wd:Q6625963 .} #occupation - novelist
  UNION
  {?item wdt:P106 wd:Q49757 .} # occupation - poet

  OPTIONAL {?item wdt:P21 ?sex }. #put the sex/gender into a variable
  OPTIONAL {?item wdt:P569 ?dob} .#put the date of birth into a variable
  OPTIONAL {?item wdt:P570 ?dod }. #put the date of death into a variable

  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }
}
LIMIT 50
"""

# Adapted query from the wikidata tutorial in notebooks/tutorial_wikidata_sparql.ipynb
query_2 = """ 
# B20 German writers OR novelists OR poets with some exammple properties
SELECT ?author ?authorLabel ?sex_genderLabel ?place_birthLabel ?place_deathLabel ?citizenshipLabel ?writing_languageLabel ?occupationLabel ?field_of_workLabel ?influenced_byLabel ?movementLabel ?membershipLabel
WHERE {

  ?author wdt:P31 wd:Q5. # humans
  
  # German citizenship OR writes in German language 
  {?author wdt:P27 wd:Q183.} # citizenship - Germany
  UNION
  {?author wdt:P6886 wd:Q188.} # writing language - German

  # occupation: writer OR novelist OR poet                                  
  {?author  wdt:P106 wd:Q36180.} #occupation - writers
  UNION
  {?author  wdt:P106 wd:Q6625963 .} #occupation - novelist
  UNION
  {?author wdt:P106 wd:Q49757 .} # occupation - poet  

  OPTIONAL {?author wdt:P21 ?sex_gender }. #put the sex/gender into a variable
  OPTIONAL {?author wdt:P19 ?place_birth} .#put the date of birth into a variable
  OPTIONAL {?author wdt:P20 ?place_death }. #put the place of death into a variable
  OPTIONAL {?author wdt:P27 ?citizenship}. #put the citizenship into a variable
  OPTIONAL {?author wdt:P6886 ?writing_language} . #put the writing language into a variable
  OPTIONAL {?author wdt:P106 ?occupation}. #put the occupation into a variable
  OPTIONAL {?author wdt:P101 ?field_of_work }. #put the field of work into a variable
  OPTIONAL {?author wdt:P737 ?influenced_by } . #put the influence into a variable; make it optional
  OPTIONAL {?author wdt:P135 ?movement } . #put the movement into a variable; make it optional
  OPTIONAL {?author wdt:P463 ?membership} . #put the membership into a variable; make it optional

  SERVICE wikibase:label { bd:serviceParam wikibase:language "en" }

}

LIMIT 50


"""


In [8]:
resp = requests.get(
    url, 
    params= {"query": query_2, "format": "json"},
    headers = {"User-Agent": user_agent, "Accept": "application/sparql-results+json"}

)

print(f"Status: {resp.status_code}")
print(f"Retry-After: {resp.headers.get('Retry-After')}")
print(f"Body preview: {resp.text[:500]}")

Status: 200
Retry-After: None
Body preview: {
  "head" : {
    "vars" : [ "author", "authorLabel", "sex_genderLabel", "place_birthLabel", "place_deathLabel", "citizenshipLabel", "writing_languageLabel", "occupationLabel", "field_of_workLabel", "influenced_byLabel", "movementLabel", "membershipLabel" ]
  },
  "results" : {
    "bindings" : [ {
      "author" : {
        "type" : "uri",
        "value" : "http://www.wikidata.org/entity/Q905"
      },
      "authorLabel" : {
        "xml:lang" : "en",
        "type" : "literal",
        "val


In [11]:
# Convert json output into pandas df

data = resp.json()

# Keys are defined by W3C SPARQL 1.1 Query Results JSON Format
# columns: item | itemLabel
rows = [
    {var: binding[var]["value"] for var in binding}
    for binding in data["results"]["bindings"]
]

df = pd.DataFrame(rows)
print(df.shape)
df.head(10)

(50, 11)


,author,authorLabel,sex_genderLabel,place_birthLabel,place_deathLabel,citizenshipLabel,writing_languageLabel,occupationLabel,field_of_workLabel,influenced_byLabel,movementLabel
0,http://www.wikidata.org/entity/Q905,Franz Kafka,male,Prague,1883-07-03T00:00:00Z,Czechoslovakia,German,film screenwriter,literature,Gustave Flaubert,existentialism
1,http://www.wikidata.org/entity/Q905,Franz Kafka,male,Prague,1883-07-03T00:00:00Z,Cisleithania,German,film screenwriter,literature,Gustave Flaubert,existentialism
2,http://www.wikidata.org/entity/Q905,Franz Kafka,male,Prague,1883-07-03T00:00:00Z,Czechoslovakia,German,writer,Prague German Literature,Gustave Flaubert,existentialism
3,http://www.wikidata.org/entity/Q905,Franz Kafka,male,Prague,1883-07-03T00:00:00Z,Cisleithania,German,writer,Prague German Literature,Gustave Flaubert,existentialism
4,http://www.wikidata.org/entity/Q905,Franz Kafka,male,Prague,1883-07-03T00:00:00Z,Czechoslovakia,German,lawyer,Prague German Literature,Gustave Flaubert,existentialism
5,http://www.wikidata.org/entity/Q905,Franz Kafka,male,Prague,1883-07-03T00:00:00Z,Cisleithania,German,lawyer,Prague German Literature,Gustave Flaubert,existentialism
6,http://www.wikidata.org/entity/Q905,Franz Kafka,male,Prague,1883-07-03T00:00:00Z,Czechoslovakia,German,poet,Prague German Literature,Gustave Flaubert,existentialism
7,http://www.wikidata.org/entity/Q905,Franz Kafka,male,Prague,1883-07-03T00:00:00Z,Cisleithania,German,poet,Prague German Literature,Gustave Flaubert,existentialism
8,http://www.wikidata.org/entity/Q905,Franz Kafka,male,Prague,1883-07-03T00:00:00Z,Czechoslovakia,German,jurist,Prague German Literature,Gustave Flaubert,existentialism
9,http://www.wikidata.org/entity/Q905,Franz Kafka,male,Prague,1883-07-03T00:00:00Z,Cisleithania,German,jurist,Prague German Literature,Gustave Flaubert,existentialism


#### 3.2) 2-Phase Architecture (Efficient for bulk extraction, [Code/Idea Source](https://gist.github.com/ArloDune/117ee69e72c1ceaae1c45818e5d646fa))

In [16]:
## 1) set configs

URL =  "https://query.wikidata.org/sparql" # phase 1 extraction
API_URL = "https://www.wikidata.org/w/api.php" # phase 2 extraction
USER_AGENT = 'kg-digital-humanities/0.1.0 (https://github.com/FabiLochner/kg-digital-humanities; fabian.lochner@uni-oldenburg.de)' #authentication

# set relevant properties of authors for the KG (property IDs -> column names)
PROPS = {
    "P27": "citizenship",
    "P6886": "writing_language",
    "P106": "occupation",
    "P21": "sex_gender",
    "P569": "date_birth",
    "P19": "place_birth",
    "P570": "date_death",
    "P69": "educated_at",
    "P101": "field_of_work",
    "P136": "genre",
    "P463": "member_of",
    "P737": "influenced_by",
    "P166": "award_received",
    "P135": "movement"
}

# properties with time values (date strings) — need different extraction logic
# all other properties in PROPS return entity QIDs
TIME_VALUED_PROPS = {"P569", "P570"} # date_birth, date_death

In [17]:
## 2) write functions 

### 1. retry helper (copied from the Source)

def parse_retry_after(header_val: str | None, default: float = 60.0) -> float:
    """Parse Retry-After header — handles both integer seconds and HTTP-date. Wikidata has 60 seconds query runtime limit.
    https://www.mediawiki.org/wiki/Wikidata_Query_Service/User_Manual#Query_limits"""
    if not header_val:
        return default
    try:
        return max(0.0, float(header_val))
    except ValueError:
        pass
    try:
        dt = parsedate_to_datetime(header_val)
        return max(0.0, (dt - datetime.datetime.now(datetime.timezone.utc)).total_seconds())
    except Exception:
        return default
    


### 2. SPARQL helpers (used in Phase 1)


def fetch_sparql(query: str) -> requests.Response:
    """Send a SPARQL SELECT query to WDQS, return raw response."""
    return requests.get(
        URL,
        params={"query": query, "format": "json"},
        headers={"User-Agent": USER_AGENT, "Accept": "application/sparql-results+json"}
    )


def parse_to_df(resp: requests.Response) -> pd.DataFrame:
    """Convert a SPARQL JSON response to a clean DataFrame (values only)."""
    data = resp.json()
    rows = [
        {var: binding[var]["value"] for var in binding}
        for binding in data["results"]["bindings"]
    ]
    return pd.DataFrame(rows)


def run_query(query: str, debug: bool = False) -> pd.DataFrame:
    """Fetch a SPARQL query and return results as a DataFrame."""
    resp = fetch_sparql(query)
    if debug:
        print(f"Status: {resp.status_code}")
        print(f"Retry-After: {resp.headers.get('Retry-After')}")
        print(f"Body preview:\n{resp.text[:300]}")
    resp.raise_for_status()
    return parse_to_df(resp)



### 3. Phase 1: collect only Wikidate IDs (QIDs) via SPARQL

def fetch_author_qids_paged(limit: int = 500, max_pages: int = 1) -> list[str]:
    """
    Lean SPARQL — no OPTIONALs, no label service, QIDs only.
    max_pages=1  →   500 QIDs (~10s,  test notebook) (500 taken from the Code/Idea Source)
    max_pages=4  → 2,000 QIDs (~2min, exploratory)
    max_pages=70 → 30 - 35.000 QIDs (full German corpus; ca 31.700 results)
    """

    all_qids = []
    for page in range(max_pages):
        offset = page * limit 
        query = f"""
        SELECT DISTINCT ?author WHERE {{ # remove duplicate authors
          ?author wdt:P31 wd:Q5. # humans
          # German citizenship OR writes in German language
          {{ ?author wdt:P27 wd:Q183. }} UNION    # citizenship - Germany
          {{ ?author wdt:P6886 wd:Q188. }}         # language written - German
          # occupation: writer OR novelist OR poet
          {{ ?author wdt:P106 wd:Q36180. }} UNION  # occupation - writer
          {{ ?author wdt:P106 wd:Q6625963. }} UNION # occupation - novelist
          {{ ?author wdt:P106 wd:Q49757. }}         # occupation - poet
        }}
        ORDER BY ?author
        LIMIT {limit}
        OFFSET {offset}
        """
        resp = fetch_sparql(query)

        # respect rate limit: wait exactly as long as the server requests
        if resp.status_code == 429:
            wait = parse_retry_after(resp.headers.get("Retry-After"))
            print(f"429 on page {page + 1} — waiting {wait:.0f}s")
            time.sleep(wait)
            resp = fetch_sparql(query)  # one retry after waiting

        # raise immediately on any other HTTP error (4xx, 5xx)
        resp.raise_for_status()

        # parse response and extract QID from each entity URI
        df = parse_to_df(resp)
        qids = [uri.split("/")[-1] for uri in df["author"].tolist()]
        all_qids.extend(qids)
        print(f"Page {page + 1}/{max_pages}: +{len(qids)} QIDs | total={len(all_qids)}")

        # fewer results than limit means this is the last page
        if len(qids) < limit:
            print("Last page reached.")
            break

        # courtesy sleep between pages to stay within the 60 CPU-s/min budget
        if page < max_pages - 1:
            time.sleep(5)

    return all_qids



def extract_time_from_claims(claims: dict, prop_id: str) -> str | None:
    """
    Extract the first valid time string for a given property (e.g. P569 = date of birth).
    Returns ISO-style string like '+1883-07-03T00:00:00Z', or None if not present.
    """
    if prop_id not in claims:
        # property not present on this entity at all
        return None
    for claim in claims[prop_id]:
        if (
            claim.get("rank") != "deprecated"           # skip superseded values
            and claim["mainsnak"].get("snaktype") == "value"  # skip 'somevalue'/'novalue'
            and claim["mainsnak"]["datavalue"]["type"] == "time"  # confirm it's a time value
        ):
            # return the raw time string from the nested value dict
            return claim["mainsnak"]["datavalue"]["value"]["time"]
    return None



def extract_qids_from_claims(claims: dict, prop_id: str) -> list[str]:
    """
    Extract all valid entity QIDs for a given property (e.g. P106 = occupation).
    Returns a list because properties can be multi-valued (e.g. multiple occupations).
    Returns empty list if property is not present or has no valid values.
    """
    if prop_id not in claims:
        # property not present on this entity at all
        return []
    return [
        claim["mainsnak"]["datavalue"]["value"]["id"]  # extract QID e.g. 'Q36180'
        for claim in claims[prop_id]
        if (
            claim.get("rank") != "deprecated"                        # skip superseded values
            and claim["mainsnak"].get("snaktype") == "value"         # skip 'somevalue'/'novalue'
            and claim["mainsnak"]["datavalue"]["type"] == "wikibase-entityid"  # confirm it's an entity ref
        )
    ]

def parse_entities(entities: dict) -> pd.DataFrame:
    """
    Convert raw wbgetentities API response to one row per author.
    Uses the global PROPS dict to determine which properties to extract.
    Time-valued properties (P569, P570) use extract_time_from_claims.
    All other properties use extract_qids_from_claims → return lists of QIDs.
    """
    rows = []
    for qid, entity in entities.items():

        # skip entities the API couldn't find (e.g. deleted or merged items)
        if "missing" in entity:
            continue

        claims = entity.get("claims", {})

        # start row with URL and English label
        row = {
            "author":      f"http://www.wikidata.org/entity/{qid}",
            "authorLabel": entity.get("labels", {}).get("en", {}).get("value", qid),
        }

        # extract each property defined in PROPS using the correct extractor
        for prop_id, col_name in PROPS.items():
            if prop_id in TIME_VALUED_PROPS:
                # date properties → single time string (e.g. '+1883-07-03T00:00:00Z')
                row[col_name] = extract_time_from_claims(claims, prop_id)
            else:
                # entity properties → list of QIDs (e.g. ['Q183', 'Q38'])
                row[col_name] = extract_qids_from_claims(claims, prop_id)

        rows.append(row)

    return pd.DataFrame(rows)

def fetch_all_entities(qids: list[str], sleep: float = 1.0) -> pd.DataFrame:
    """
    Phase 2 orchestrator: batch QIDs into groups of 50, call wbgetentities
    for each batch, parse results with parse_entities().
    sleep=1.0s between batches is sufficient — wbgetentities runs on a
    separate CDN-cached quota, not the SPARQL 60 CPU-s/min budget.
    """
    if not qids:
        raise ValueError("QIDs list is empty - Phase 1 returned no results")
    # split full QID list into batches of 50 (API maximum per request)
    batches = [qids[i:i + 50] for i in range(0, len(qids), 50)]
    dfs = []

    for i, batch in enumerate(batches):
        resp = requests.get(
            API_URL,
            params={
                "action":    "wbgetentities",
                "ids":       "|".join(batch),  # pipe-separated QIDs e.g. 'Q905|Q34660'
                "props":     "labels|claims",  # only fetch labels and property claims
                "languages": "en",
                "format":    "json",
            },
            headers={"User-Agent": USER_AGENT}
        )

        # respect rate limit: wait exactly as long as the server requests
        if resp.status_code == 429:
            wait = parse_retry_after(resp.headers.get("Retry-After"))
            print(f"429 on batch {i + 1} — waiting {wait:.0f}s")
            time.sleep(wait)
            resp = requests.get(  # one retry after waiting
                API_URL,
                params={
                    "action":    "wbgetentities",
                    "ids":       "|".join(batch),
                    "props":     "labels|claims",
                    "languages": "en",
                    "format":    "json",
                },
                headers={"User-Agent": USER_AGENT}
            )

        # raise immediately on any other HTTP error (4xx, 5xx)
        resp.raise_for_status()

        # parse the batch response into one row per author using PROPS
        dfs.append(parse_entities(resp.json()["entities"]))
        print(f"Batch {i + 1}/{len(batches)} done")

        # courtesy sleep between batches to avoid hammering the API
        time.sleep(sleep)

    # combine all batch DataFrames into one
    return pd.concat(dfs, ignore_index=True)

In [ ]:
## 3) run the 2 phase architecture code 

# Phase 1 — collect QIDs
qids = fetch_author_qids_paged(limit=500, max_pages=1)  # change max_pages to scale up

# Phase 2 — fetch properties for each QID
authors_df = fetch_all_entities(qids)


Page 1/1: +500 QIDs | total=500
Batch 1/10 done
Batch 2/10 done
Batch 3/10 done
Batch 4/10 done
Batch 5/10 done
Batch 6/10 done
Batch 7/10 done
Batch 8/10 done
Batch 9/10 done
Batch 10/10 done
(500, 16)


,author,authorLabel,citizenship,writing_language,occupation,sex_gender,date_birth,place_birth,date_death,educated_at,field_of_work,genre,member_of,influenced_by,award_received,movement
0,http://www.wikidata.org/entity/Q1000002,Claus Hammel,[Q183],[],"[Q214917, Q1930187, Q36180, Q28389, Q17337766]",[Q6581097],+1932-12-04T00:00:00Z,[Q559010],+1990-04-12T00:00:00Z,[],[],[],[],[],"[Q1351858, Q12662961, Q449353, Q1605666]",[]
1,http://www.wikidata.org/entity/Q1000087,Horst Baier,[Q183],[],"[Q2306091, Q1622272, Q36180, Q39631, Q49757]",[Q6581097],+1933-03-26T00:00:00Z,[Q14960],+2017-12-02T00:00:00Z,[],[],[],[Q23785357],[],[],[]
2,http://www.wikidata.org/entity/Q1000180,Thilo Koch,[Q183],[],"[Q1930187, Q114347127, Q15980158, Q36180]",[Q6581097],+1920-09-20T00:00:00Z,[Q2814],+2006-09-12T00:00:00Z,[],[],[],[],[],"[Q445673, Q10905334]",[]
3,http://www.wikidata.org/entity/Q100029,Otto Hermann von Vietinghoff,"[Q183, Q34266]",[],"[Q36180, Q82955, Q47064, Q104680, Q1501800, Q1...",[Q6581097],+1722-12-03T00:00:00Z,[Q1773],+1792-06-24T00:00:00Z,[],[],[],[],[],[],[]
4,http://www.wikidata.org/entity/Q100031,Kaspar von Stieler,[Q183],[Q188],"[Q36180, Q49757, Q14467526, Q14972848, Q208265...",[Q6581097],+1632-08-02T00:00:00Z,[Q1729],+1707-06-24T00:00:00Z,[],[],[],[Q559186],[],[],[]


In [20]:
print(authors_df.shape)
authors_df.head(30)

(500, 16)


,author,authorLabel,citizenship,writing_language,occupation,sex_gender,date_birth,place_birth,date_death,educated_at,field_of_work,genre,member_of,influenced_by,award_received,movement
0,http://www.wikidata.org/entity/Q1000002,Claus Hammel,[Q183],[],"[Q214917, Q1930187, Q36180, Q28389, Q17337766]",[Q6581097],+1932-12-04T00:00:00Z,[Q559010],+1990-04-12T00:00:00Z,[],[],[],[],[],"[Q1351858, Q12662961, Q449353, Q1605666]",[]
1,http://www.wikidata.org/entity/Q1000087,Horst Baier,[Q183],[],"[Q2306091, Q1622272, Q36180, Q39631, Q49757]",[Q6581097],+1933-03-26T00:00:00Z,[Q14960],+2017-12-02T00:00:00Z,[],[],[],[Q23785357],[],[],[]
2,http://www.wikidata.org/entity/Q1000180,Thilo Koch,[Q183],[],"[Q1930187, Q114347127, Q15980158, Q36180]",[Q6581097],+1920-09-20T00:00:00Z,[Q2814],+2006-09-12T00:00:00Z,[],[],[],[],[],"[Q445673, Q10905334]",[]
3,http://www.wikidata.org/entity/Q100029,Otto Hermann von Vietinghoff,"[Q183, Q34266]",[],"[Q36180, Q82955, Q47064, Q104680, Q1501800, Q1...",[Q6581097],+1722-12-03T00:00:00Z,[Q1773],+1792-06-24T00:00:00Z,[],[],[],[],[],[],[]
4,http://www.wikidata.org/entity/Q100031,Kaspar von Stieler,[Q183],[Q188],"[Q36180, Q49757, Q14467526, Q14972848, Q208265...",[Q6581097],+1632-08-02T00:00:00Z,[Q1729],+1707-06-24T00:00:00Z,[],[],[],[Q559186],[],[],[]
5,http://www.wikidata.org/entity/Q100035,Franz von Gaudy,[Q183],[Q188],"[Q49757, Q333634, Q6625963, Q36180]",[Q6581097],+1800-01-01T00:00:00Z,[Q4024],+1840-01-01T00:00:00Z,[Q318186],[],[],[],[],[],[]
6,http://www.wikidata.org/entity/Q1000687,Karl Otto Henseling,[Q183],[],"[Q593644, Q36180, Q15980158]",[Q6581097],+1945-06-26T00:00:00Z,[Q64],+2011-01-28T00:00:00Z,[],[],[],[],[],[],[]
7,http://www.wikidata.org/entity/Q100073,Leberecht Migge,[Q183],[],"[Q2815948, Q36180, Q15432439, Q482980]",[Q6581097],+1881-03-20T00:00:00Z,[Q1792],+1935-05-30T00:00:00Z,[],[],[],[Q160438],[Q85834],[],[]
8,http://www.wikidata.org/entity/Q1000916,Albert Károly Festetics,[Q28],[Q188],"[Q49757, Q1930187, Q9363889]",[Q6581097],+1790-00-00T00:00:00Z,[],+1880-00-00T00:00:00Z,[],[],[],[],[],[],[]
9,http://www.wikidata.org/entity/Q100136,Otto Böckel,[Q183],[],"[Q36180, Q82955, Q1371378, Q182436, Q486839, Q...",[Q6581097],+1859-07-02T00:00:00Z,[Q1794],+1923-01-01T00:00:00Z,[Q155354],[],[],[Q26836255],[],[],[]
